# 📈 S&P 500 指數預測系統
## 課程：114-2 ML & DL

**目標：** 使用 XGBoost 與 Random Forest 比較預測 S&P 500 收盤指數，並輸出 MSE 評估與視覺化結果。

---

### ⚠️ 學術倫理聲明
| 規範 | 說明 |
|------|------|
| 資料來源 | `yfinance` 下載 `^GSPC`（含 `^VIX`） |
| 股利還原 | `auto_adjust=True`，使用 Adj Close 避免股利拖曳偏差 |
| 時間範圍 | 2021-01-01 ～ 2025-12-31 |
| 訓練集 | 2021 ～ 2024（**嚴禁接觸測試集**） |
| 測試集 | 2025 全年（**只用於最終評估**） |
| 切分方式 | 依時間順序，**嚴禁隨機洗牌** |
| 防洩漏 | 所有特徵為落後型/滾動型，不含未來資料 |

---
## 🔧 Cell 1 — 環境前置與套件安裝（必須先執行）

### ⚠️ macOS 使用者：XGBoost 需要 OpenMP（必做）

> 若你在 macOS 上使用 VSCode 或 Jupyter，XGBoost 需要 `libomp` 才能載入。
> 請先在 **macOS 終端機（Terminal.app）** 執行以下指令安裝：
>
> ```bash
> # 1. 先安裝 Homebrew（若還沒有）
> /bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"
>
> # 2. 安裝 OpenMP runtime（XGBoost 在 macOS 的依賴）
> brew install libomp
> ```
>
> 安裝完成後，**重新啟動 VSCode**，再繼續執行 Cell 1。

---

### 為什麼用 `%pip` 而不是 `!pip`？
> - `!pip install`：在系統 shell 執行，套件可能裝到**其他 Python 環境**，導致 `import` 找不到
> - `%pip install`：Jupyter 魔術指令，**直接裝到目前 Kernel 的 Python 環境**，VSCode 必用此方式
>
> 安裝完成後若仍報 `ModuleNotFoundError`，請點右上角 **Select Kernel** 重新選擇 Python 版本再重試。

In [ ]:
# ── 路徑設定（執行前請先確認工作目錄）────────────────────────
# 建議從專案根目錄（RA7141236_HW_1/）啟動 Jupyter：
#   cd RA7141236_HW_1 && jupyter notebook
# 若你從其他位置啟動，請修改下方 BASE_DIR
import os
from pathlib import Path

# 自動偵測：如果 CWD 是 notebooks/，BASE_DIR 往上一層
CWD = Path(os.getcwd())
BASE_DIR = CWD.parent if CWD.name == 'notebooks' else CWD

REPORTS_DIR = BASE_DIR / 'reports'
DATA_DIR    = BASE_DIR / 'data'
REPORTS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

print(f'📁 BASE_DIR    = {BASE_DIR}')
print(f'📊 REPORTS_DIR = {REPORTS_DIR}')
print(f'📋 DATA_DIR    = {DATA_DIR}')

In [ ]:
# ============================================================
# 【必讀】macOS 使用者：若 XGBoost 載入失敗，請先在 Terminal 執行：
#   brew install libomp
# 安裝完後重新啟動 VSCode，再繼續執行此格
# ============================================================

# ✅ 使用 %pip 確保套件裝到目前 Kernel 使用的 Python 環境
%pip install yfinance scikit-learn xgboost pandas numpy matplotlib scipy --quiet

print('✅ 套件安裝完成！請繼續執行 Cell 2')
print('⚠️  若 XGBoost 報錯（macOS）：在 Terminal 執行 brew install libomp 後重啟 VSCode')
print('⚠️  若其他 import 仍報錯：點右上角 Select Kernel → 選正確 Python → 重執行此格')

---
## 📦 Cell 2 — 匯入套件與全域設定
> 修改此格的常數，即可調整實驗的時間範圍與切分點。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import yfinance as yf

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# 中文字體設定（依作業系統自動選擇）
# macOS → Arial Unicode MS；Windows → Microsoft JhengHei；Linux → 回退 DejaVu Sans
if sys.platform == 'darwin':
    plt.rcParams['font.family'] = ['Arial Unicode MS', 'DejaVu Sans']
elif sys.platform == 'win32':
    plt.rcParams['font.family'] = ['Microsoft JhengHei', 'DejaVu Sans']
else:
    plt.rcParams['font.family'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 確保負號正確顯示

# ========== 全域常數（修改此處調整實驗設定） ==========
START_DATE = '2021-01-01'  # 資料起始日
END_DATE   = '2025-12-31'  # 資料結束日
SPLIT_DATE = '2025-01-01'  # 切分點：此日期之前為訓練集

print('✅ 套件匯入完成')
print(f'   Python 版本：{sys.version.split()[0]}')
print(f'   訓練集：{START_DATE} ～ 2024-12-31')
print(f'   測試集：{SPLIT_DATE} ～ {END_DATE}')

---
## 📥 Cell 3 — 資料下載
- **`^GSPC`**：S&P 500 指數，設定 `auto_adjust=True` 自動還原除息股價
- **`^VIX`**：CBOE 恐慌指數，反映市場對未來波動的預期

In [ ]:
def download_data(ticker, start, end, auto_adjust=True):
    """
    從 Yahoo Finance 下載歷史資料。
    auto_adjust=True → Close 欄自動替換為 Adj Close（還原除息影響）
    """
    print(f'  下載 {ticker} ({start} ~ {end})...')
    df = yf.download(ticker, start=start, end=end,
                     auto_adjust=auto_adjust, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[~df.index.duplicated(keep='last')].sort_index()
    print(f'  完成！共 {len(df)} 筆交易日資料')
    return df


def clean_data(df, name='資料'):
    """
    資料清洗：前向填補（假日缺失）→ 後向填補（邊界保護）
    不刪除任何資料點，以保留時間序列連續性。
    """
    missing = df.isnull().sum().sum()
    if missing > 0:
        df = df.ffill().bfill()
        print(f'  {name}：填補了 {missing} 個缺失值')
    else:
        print(f'  {name}：無缺失值 ✓')
    return df


print('【Step 1/9】下載原始資料')
gspc_raw = download_data('^GSPC', START_DATE, END_DATE, auto_adjust=True)
vix_raw  = download_data('^VIX',  START_DATE, END_DATE, auto_adjust=False)

print('\n【Step 2/9】清洗資料')
gspc_clean = clean_data(gspc_raw, '^GSPC')
vix_clean  = clean_data(vix_raw,  '^VIX')

print('\n▶ S&P 500 資料預覽（前 3 筆）：')
display(gspc_clean.head(3))

---
## 🛠️ Cell 4 — 特徵工程
### 防洩漏核心原則
所有特徵在時間點 `t` 只使用 `t-1` 及更早的資料：
- **落後型**：`shift(1)`, `shift(5)`, `shift(10)`
- **滾動型**：先 `shift(1)` 再 `rolling()` — 確保均線不含當日收盤
- **VIX 特徵**：全部 `shift(1)`
- **Spike 閾值**：只用訓練集統計量

In [ ]:
def compute_rsi(series, period=14):
    """
    RSI（相對強弱指數）= 100 - 100/(1+RS)
    RS = 平均漲幅 ÷ 平均跌幅（Wilder 指數平滑法）
    輸入的 series 必須已 shift(1)，確保不含當日資訊。
    """
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.ewm(com=period-1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period-1, min_periods=period).mean()
    rs = avg_gain / (avg_loss + 1e-10)
    return 100 - (100 / (1 + rs))


def build_features(gspc_df, vix_df):
    """
    建構 18 個輸入特徵 + 1 個預測目標（次日 Adj Close）。
    所有特徵皆為落後型或滾動型，確保 F(t) 只使用 t-1 以前的資料。
    """
    df = gspc_df[['Close','High','Low','Volume']].copy()
    df.columns = ['adj_close','high','low','volume']

    # 合併 VIX（以 GSPC 交易日為主，left join + ffill 補齊）
    df = df.join(vix_df['Close'].rename('vix_raw'), how='left')
    df['vix_raw'] = df['vix_raw'].ffill()

    # 落後型特徵
    df['lag_1']  = df['adj_close'].shift(1)
    df['lag_5']  = df['adj_close'].shift(5)
    df['lag_10'] = df['adj_close'].shift(10)

    # 滾動均線（先 shift(1) 確保不含當日收盤）
    c1 = df['adj_close'].shift(1)
    df['ma_5']          = c1.rolling(5,  min_periods=5).mean()
    df['ma_20']         = c1.rolling(20, min_periods=20).mean()
    df['ma_60']         = c1.rolling(60, min_periods=60).mean()
    df['volatility_10'] = c1.rolling(10, min_periods=10).std()

    # 成交量變化率
    v1 = df['volume'].shift(1);  v2 = df['volume'].shift(2)
    df['volume_change'] = (v1 - v2) / (v2 + 1e-10)

    # RSI（使用已落後的 c1）
    df['rsi_14'] = compute_rsi(c1, period=14)

    # 週期性特徵
    df['day_of_week'] = df.index.dayofweek
    df['month']       = df.index.month

    # 日報酬率（pct_change 內建 shift(1)）
    df['daily_return'] = df['adj_close'].pct_change(1)

    # VIX 特徵（全部使用前一日 VIX）
    vix1              = df['vix_raw'].shift(1)
    df['vix_close']   = vix1
    df['vix_ma_5']    = vix1.rolling(5, min_periods=5).mean()
    df['vix_change']  = vix1.pct_change(1)
    df['vix_regime']  = (vix1 > 20).astype(int)  # 1=高波動, 0=低波動

    # 預測目標：明日收盤價
    df['target'] = df['adj_close'].shift(-1)

    df = df.drop(columns=['vix_raw'])
    n_before = len(df)
    df = df.dropna()
    print(f'  移除 {n_before - len(df)} 筆 NaN 行，最終：{len(df)} 筆 × {len(df.columns)} 欄')
    return df


print('【Step 3/9】建構特徵矩陣')
df_features = build_features(gspc_clean, vix_clean)
print('\n▶ 特徵矩陣欄位：', list(df_features.columns))
display(df_features.head(3))

---
## ✂️ Cell 5 — 時間序列切分 + 大波動標記
> ⚠️ 此格含關鍵防洩漏斷言：若切分錯誤會立即拋出例外，確保訓練集不含 2025 年資料。

In [ ]:
print('【Step 4/9】時間序列切分（依時間順序，嚴禁隨機洗牌）')

train_mask = df_features.index < SPLIT_DATE
test_mask  = df_features.index >= SPLIT_DATE
train_df   = df_features[train_mask]
test_df    = df_features[test_mask]

# 安全斷言：任一失敗即表示資料洩漏，必須修正
assert train_df.index.max() < pd.Timestamp(SPLIT_DATE), '❌ 訓練集含 2025 年資料！'
assert test_df.index.min() >= pd.Timestamp(SPLIT_DATE), '❌ 測試集含 2024 年以前資料！'

print(f'  訓練集：{train_df.index.min().date()} ～ {train_df.index.max().date()} （{len(train_df)} 天）✅')
print(f'  測試集：{test_df.index.min().date()} ～ {test_df.index.max().date()} （{len(test_df)} 天）✅')

# 標記大波動（閾值只用訓練集計算，防止測試集洩漏）
print('\n【Step 5/9】標記大幅波動日')

mu        = train_df['daily_return'].mean()
sigma     = train_df['daily_return'].std()
threshold = 2.0 * sigma
print(f'  訓練集統計：μ={mu:.4%}, σ={sigma:.4%}, 閾值=|日報酬| > {(abs(mu)+threshold):.4%}')

# 套用至全資料（測試集沿用訓練集統計量，不重新計算）
df_features['is_spike']        = (df_features['daily_return'].abs() > (abs(mu)+threshold)).astype(int)
df_features['spike_direction'] = 0
df_features.loc[df_features['daily_return'] >  (abs(mu)+threshold), 'spike_direction'] =  1
df_features.loc[df_features['daily_return'] < -(abs(mu)+threshold), 'spike_direction'] = -1

train_df = df_features[train_mask]
test_df  = df_features[test_mask]

test_spikes = test_df[test_df['is_spike'] == 1][['daily_return','spike_direction']].copy()
test_spikes['daily_return'] = test_spikes['daily_return'].map('{:.3%}'.format)
print(f'\n  2025 年大波動日（共 {len(test_spikes)} 天）：')
display(test_spikes)

---
## 🤖 Cell 6 — 訓練 XGBoost
使用 `TimeSeriesSplit`（5 折）在訓練集內部評估，**不接觸測試集**。

In [ ]:
# 定義 18 個特徵欄位
FEATURE_COLS = [
    'lag_1', 'lag_5', 'lag_10',
    'ma_5', 'ma_20', 'ma_60',
    'volatility_10', 'volume_change', 'rsi_14',
    'day_of_week', 'month', 'daily_return',
    'vix_close', 'vix_ma_5', 'vix_change', 'vix_regime',
    'is_spike', 'spike_direction'
]

print(f'【Step 6/9】準備特徵矩陣（{len(FEATURE_COLS)} 個特徵）')
X_train = train_df[FEATURE_COLS];  y_train = train_df['target']
X_test  = test_df[FEATURE_COLS];   y_test  = test_df['target']
print(f'  訓練：{X_train.shape}  測試：{X_test.shape}')

tscv = TimeSeriesSplit(n_splits=5)  # 5 折時間序列交叉驗證

print('\n【Step 7a/9】訓練 XGBoost')
# 超參數說明：
#   max_depth=4      : 淺樹，防止在金融噪音資料中過擬合
#   learning_rate=0.05: 小步長，搭配多樹緩慢收斂提升泛化
#   subsample=0.8    : 每棵樹用 80% 樣本，增加多樣性
#   colsample_bytree=0.8: 每棵樹用 80% 特徵，降低共線性
XGB_PARAMS = dict(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0,
    objective='reg:squarederror', random_state=42, n_jobs=-1
)

xgb_cv_scores = []
for i, (tr, val) in enumerate(tscv.split(X_train)):
    m = XGBRegressor(**XGB_PARAMS)
    m.fit(X_train.iloc[tr], y_train.iloc[tr], verbose=False)
    mse = mean_squared_error(y_train.iloc[val], m.predict(X_train.iloc[val]))
    xgb_cv_scores.append(mse)
    print(f'  Fold {i+1}: MSE = {mse:,.2f}')

print(f'  CV 平均 MSE = {np.mean(xgb_cv_scores):,.2f} (±{np.std(xgb_cv_scores):,.2f})')
xgb_model = XGBRegressor(**XGB_PARAMS)
xgb_model.fit(X_train, y_train, verbose=False)  # 用完整訓練集訓練最終模型
print('  XGBoost 訓練完成 ✅')

---
## 🌲 Cell 7 — 訓練 Random Forest

In [ ]:
print('【Step 7b/9】訓練 Random Forest')
# 超參數說明：
#   n_estimators=300    : 300 棵獨立樹，集成效果穩定
#   max_depth=6         : 比 XGBoost 略深（各樹獨立不會累積誤差）
#   min_samples_split=20: 節點分裂最少需要 20 個樣本
#   max_features='sqrt' : 每次分裂用特徵數的平方根（標準設定）
RF_PARAMS = dict(
    n_estimators=300, max_depth=6,
    min_samples_split=20, min_samples_leaf=10,
    max_features='sqrt', bootstrap=True,
    random_state=42, n_jobs=-1
)

rf_cv_scores = []
for i, (tr, val) in enumerate(tscv.split(X_train)):
    m = RandomForestRegressor(**RF_PARAMS)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    mse = mean_squared_error(y_train.iloc[val], m.predict(X_train.iloc[val]))
    rf_cv_scores.append(mse)
    print(f'  Fold {i+1}: MSE = {mse:,.2f}')

print(f'  CV 平均 MSE = {np.mean(rf_cv_scores):,.2f} (±{np.std(rf_cv_scores):,.2f})')
rf_model = RandomForestRegressor(**RF_PARAMS)
rf_model.fit(X_train, y_train)
print('  Random Forest 訓練完成 ✅')

---
## 📊 Cell 8 — 測試集評估（主指標：MSE）

In [ ]:
print('【Step 8/9】測試集評估（2025 全年，不接觸訓練集）')

xgb_pred = xgb_model.predict(X_test)
rf_pred  = rf_model.predict(X_test)

def get_metrics(y_true, y_pred, name):
    """
    計算四項評估指標：
    - MSE  ：均方誤差（主要指標，越低越好）
    - RMSE ：根均方誤差（與 S&P 500 點數同單位）
    - MAE  ：平均絕對誤差（對極端值較穩健）
    - MAPE ：平均絕對百分比誤差（%，跨市場可比較）
    """
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(y_true.values - y_pred))
    mape = np.mean(np.abs((y_true.values - y_pred) / (y_true.values + 1e-10))) * 100
    return {'模型': name, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE(%)': mape}

xgb_m = get_metrics(y_test, xgb_pred, 'XGBoost')
rf_m  = get_metrics(y_test, rf_pred,  'Random Forest')

# 彩色比較表（MSE 最小欄標綠色）
# 用括號包覆鏈式呼叫，避免反斜線續行在 Notebook 中不穩定的問題
metrics_df = pd.DataFrame([xgb_m, rf_m]).set_index('模型')
styled = (
    metrics_df.style
    .format({'MSE': '{:,.2f}', 'RMSE': '{:,.2f}', 'MAE': '{:,.2f}', 'MAPE(%)': '{:.4f}'})
    .highlight_min(subset=['MSE', 'RMSE', 'MAE', 'MAPE(%)'], color='#d4edda')
)
display(styled)

# 儲存 CSV（可用 Excel 開啟）
pd.DataFrame([xgb_m, rf_m]).to_csv(
    str(DATA_DIR / 'model_comparison.csv'), index=False, float_format='%.6f', encoding='utf-8-sig'
)
winner = 'XGBoost' if xgb_m['MSE'] < rf_m['MSE'] else 'Random Forest'
print(f'\n🏆 MSE 最低（表現最佳）模型：{winner}')
print('📋 比較表已儲存 → model_comparison.csv')

---
## 🎨 Cell 9 — 視覺化：預測對照 + VIX + 誤差分析（三合一）

In [ ]:
print('【Step 9a/9】繪製預測對照圖（圖一至圖三）')

# 色彩方案
C_ACTUAL='#58a6ff'; C_XGB='#f78166'; C_RF='#3fb950'
C_SPIKE='#ff7b72'; C_VIX='#d2a8ff'; C_GRID='#21262d'; C_TEXT='#c9d1d9'
BG='#0d1117'; BG_P='#161b22'

test_dates   = test_df.index
test_spike_d = test_df[test_df['is_spike'] == 1].index
vix_aligned  = vix_clean['Close'].reindex(test_dates).ffill()

fig, axes = plt.subplots(3, 1, figsize=(18, 18))
fig.patch.set_facecolor(BG)

# ── 圖一：實際值 vs. 預測值（含大波動垂直虛線）──
ax1 = axes[0]; ax1.set_facecolor(BG_P)
ax1.plot(test_dates, y_test,   color=C_ACTUAL, lw=2.0, label='實際收盤價（Adj Close）')
ax1.plot(test_dates, xgb_pred, color=C_XGB, lw=1.5, ls='--', label='XGBoost 預測')
ax1.plot(test_dates, rf_pred,  color=C_RF,  lw=1.5, ls=':',  label='Random Forest 預測')
for d in test_spike_d:
    ax1.axvline(d, color=C_SPIKE, alpha=0.55, lw=1.2, ls='--')
sp = mpatches.Patch(color=C_SPIKE, alpha=0.6, label=f'大波動（{len(test_spike_d)} 天）')
ax1.set_title(
    f'S&P 500 指數預測（2025 年）\n'
    f'XGBoost RMSE={xgb_m["RMSE"]:.2f}pt MAPE={xgb_m["MAPE(%)"]:.3f}%   │   '
    f'RF RMSE={rf_m["RMSE"]:.2f}pt MAPE={rf_m["MAPE(%)"]:.3f}%',
    color=C_TEXT, fontsize=12, pad=12)
ax1.set_ylabel('指數點數', color=C_TEXT); ax1.tick_params(colors=C_TEXT)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m')); ax1.grid(color=C_GRID, lw=0.8)
h,l = ax1.get_legend_handles_labels()
ax1.legend(h+[sp], l+[sp.get_label()], facecolor='#21262d', labelcolor=C_TEXT, fontsize=9)
[s.set_edgecolor(C_GRID) for s in ax1.spines.values()]

# ── 圖二：VIX × S&P 500 雙軸（高波動區間塗色）──
ax2 = axes[1]; ax2.set_facecolor(BG_P)
ax2.plot(test_dates, y_test, color=C_ACTUAL, lw=1.8, label='S&P 500（左軸）')
ax2.set_ylabel('S&P 500 指數', color=C_ACTUAL); ax2.tick_params(axis='y', colors=C_ACTUAL)
ax2.tick_params(axis='x', colors=C_TEXT)
ax2r = ax2.twinx()
ax2r.plot(test_dates, vix_aligned.values, color=C_VIX, lw=1.4, alpha=0.85, label='VIX（右軸）')
ax2r.set_ylabel('VIX 恐慌指數', color=C_VIX); ax2r.tick_params(axis='y', colors=C_VIX)
ax2.fill_between(test_dates, y_test.min()*0.97, y_test.max()*1.03,
                 where=(vix_aligned.values>20), color=C_SPIKE, alpha=0.1, label='高波動(VIX>20)')
ax2.set_title('VIX 恐慌指數 × S&P 500 走勢（2025 年）', color=C_TEXT, fontsize=12, pad=12)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m')); ax2.grid(color=C_GRID, lw=0.8)
ln1,lb1=ax2.get_legend_handles_labels(); ln2,lb2=ax2r.get_legend_handles_labels()
ax2.legend(ln1+ln2, lb1+lb2, facecolor='#21262d', labelcolor=C_TEXT, fontsize=9)
[s.set_edgecolor(C_GRID) for s in ax2.spines.values()]

# ── 圖三：誤差分析（大波動日三角形標記）──
ax3 = axes[2]; ax3.set_facecolor(BG_P)
xgb_err = y_test.values - xgb_pred;  rf_err = y_test.values - rf_pred
ax3.plot(test_dates, xgb_err, color=C_XGB, lw=1.2, alpha=0.85, label='XGBoost 誤差')
ax3.plot(test_dates, rf_err,  color=C_RF,  lw=1.2, alpha=0.85, label='RF 誤差')
ax3.axhline(0, color=C_TEXT, lw=1.0, alpha=0.5)
for d in test_spike_d:
    if d in test_dates:
        ax3.scatter(d, xgb_err[test_dates.get_loc(d)], color=C_SPIKE, zorder=5, s=50, marker='^')
ax3.set_title('預測誤差分析（▲=大波動事件日）', color=C_TEXT, fontsize=12, pad=12)
ax3.set_ylabel('誤差（點數）', color=C_TEXT); ax3.tick_params(colors=C_TEXT)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m')); ax3.grid(color=C_GRID, lw=0.8)
ax3.legend(facecolor='#21262d', labelcolor=C_TEXT, fontsize=9)
[s.set_edgecolor(C_GRID) for s in ax3.spines.values()]

plt.tight_layout(pad=3.0)
plt.savefig('sp500_results.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('📊 sp500_results.png 已儲存')

---
## 🔬 Cell 10 — 特徵重要性排名

In [ ]:
print('【Step 9b/9】繪製特徵重要性圖')
TOP_N = 12
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor(BG)

for ax, model, title, cbar in zip(
    axes,
    [xgb_model, rf_model],
    ['XGBoost 特徵重要性（前 12）', 'Random Forest 特徵重要性（前 12）'],
    [C_XGB, C_RF]
):
    ax.set_facecolor(BG_P)
    imp = model.feature_importances_
    idx = np.argsort(imp)[::-1][:TOP_N]
    names = [FEATURE_COLS[i] for i in idx]
    vals  = imp[idx]

    bars = ax.barh(range(TOP_N), vals[::-1], color=cbar, alpha=0.85)
    ax.set_yticks(range(TOP_N)); ax.set_yticklabels(names[::-1], color=C_TEXT, fontsize=10)
    ax.set_xlabel('重要性分數', color=C_TEXT)
    ax.set_title(title, color=C_TEXT, fontsize=12, pad=10)
    ax.tick_params(axis='x', colors=C_TEXT); ax.grid(color=C_GRID, lw=0.8, axis='x')
    # VIX/Spike 特徵特殊標色
    for bi, n in enumerate(names[::-1]):
        if 'vix'   in n: bars[bi].set_color(C_VIX);   bars[bi].set_alpha(1.0)
        if 'spike' in n: bars[bi].set_color(C_SPIKE);  bars[bi].set_alpha(1.0)
    [s.set_edgecolor(C_GRID) for s in ax.spines.values()]

fig.legend(
    handles=[mpatches.Patch(color=C_VIX, label='VIX 相關特徵'),
             mpatches.Patch(color=C_SPIKE, label='大波動標記特徵')],
    loc='lower center', ncol=2, facecolor='#21262d', labelcolor=C_TEXT,
    fontsize=10, bbox_to_anchor=(0.5, -0.03)
)
plt.suptitle('模型特徵重要性比較', color=C_TEXT, fontsize=14, y=1.01)
plt.tight_layout(pad=3.0)
plt.savefig('sp500_feature_importance.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('📊 sp500_feature_importance.png 已儲存')

---
## ✅ Cell 11 — 執行總結

In [ ]:
print('=' * 60)
print('  S&P 500 預測系統 — 執行完成！')
print('=' * 60)
print(f'  訓練集：{train_df.index.min().date()} ～ {train_df.index.max().date()} ({len(train_df)} 天)')
print(f'  測試集：{test_df.index.min().date()} ～ {test_df.index.max().date()} ({len(test_df)} 天)')
print()
print(f'  XGBoost      MSE = {xgb_m["MSE"]:>12,.2f}   MAPE = {xgb_m["MAPE(%)"]:.3f}%')
print(f'  RandomForest MSE = {rf_m["MSE"]:>12,.2f}   MAPE = {rf_m["MAPE(%)"]:.3f}%')
print(f'  🏆 勝出模型：{winner}')
print()
print('  輸出檔案：')
print('    📊 sp500_results.png           （預測對照 + VIX + 誤差）')
print('    📊 sp500_feature_importance.png（特徵重要性排名）')
print('    📋 model_comparison.csv        （數值比較表，可用 Excel 開啟）')
print('=' * 60)